# 03 — BiLSTM + Attention Model

## Theory

### What is Attention?
In standard RNNs/LSTMs, the entire input sequence is compressed into a fixed-size
hidden state — like summarizing a book into a single sentence. This bottleneck means
important early tokens can be "forgotten" by the time the model processes the end.

**Attention** solves this by allowing the model to look back at ALL hidden states
and learn which ones matter most for the current prediction:
```
score(h_t) = v^T * tanh(W * h_t)   ← learned importance
alpha_t    = softmax(scores)         ← normalized weights
context    = Σ alpha_t * h_t         ← weighted sum
```

### Why BiLSTM?
A Bidirectional LSTM processes the sequence left-to-right AND right-to-left,
giving each token access to both past and future context — important for
understanding negation ("NOT fraudulent") and qualifiers at sentence boundaries.

Prerequisites: `python src/data_prep.py`

In [ ]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import torch
import joblib
from pathlib import Path

from src.attention_model import (
    build_vocab, save_vocab, load_vocab,
    train_attention_model, predict_with_attention,
    get_attention_for_text, BiLSTMAttention,
)
from src.data_prep import compute_class_weights
from src.evaluate import evaluate_model, plot_confusion_matrix, plot_training_curves

sns.set_theme(style='whitegrid')
BASE_DIR = Path('..')
DATA_DIR  = BASE_DIR / 'data' / 'processed'
MODELS_DIR = BASE_DIR / 'models'
RESULTS_DIR = BASE_DIR / 'results'

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Data

In [ ]:
train_df = pd.read_csv(DATA_DIR / 'train.csv')
val_df   = pd.read_csv(DATA_DIR / 'val.csv')
test_df  = pd.read_csv(DATA_DIR / 'test.csv')

le_product = joblib.load(MODELS_DIR / 'label_encoder_product.joblib')
label_names = list(le_product.classes_)
NUM_CLASSES = len(label_names)

train_texts  = train_df['narrative_clean'].tolist()
train_labels = train_df['product_encoded'].tolist()
val_texts    = val_df['narrative_clean'].tolist()
val_labels   = val_df['product_encoded'].tolist()
test_texts   = test_df['narrative_clean'].tolist()
test_labels  = test_df['product_encoded'].tolist()

print(f'Classes: {NUM_CLASSES} | Train: {len(train_texts):,} | Val: {len(val_texts):,} | Test: {len(test_texts):,}')

## 2. Build Vocabulary

In [ ]:
vocab_path = MODELS_DIR / 'vocab.json'

if vocab_path.exists():
    vocab = load_vocab(vocab_path)
    print(f'Loaded existing vocabulary: {len(vocab):,} words.')
else:
    vocab = build_vocab(train_df['narrative_clean'], max_vocab=50000, min_freq=3)
    save_vocab(vocab, vocab_path)
    print(f'Built and saved vocabulary: {len(vocab):,} words.')

## 3. Compute Class Weights

In [ ]:
class_weights = compute_class_weights(np.array(train_labels))

print('Class weights (higher = rarer class):')
for name, w in zip(label_names, class_weights):
    print(f'  {name[:35]:<35}: {w:.3f}')

## 4. Train BiLSTM + Attention

In [ ]:
model, history = train_attention_model(
    train_texts=train_texts,
    train_labels=train_labels,
    val_texts=val_texts,
    val_labels=val_labels,
    vocab=vocab,
    num_classes=NUM_CLASSES,
    class_weights=class_weights,
    max_len=256,
    embed_dim=128,
    hidden_dim=128,
    batch_size=64,
    learning_rate=1e-3,
    weight_decay=1e-4,
    num_epochs=15,
    patience=3,
    device=DEVICE,
)
print('Training complete.')

## 5. Training Curves

In [ ]:
plot_training_curves(history, model_name='BiLSTM_Attention')
from IPython.display import Image
Image(str(RESULTS_DIR / 'training_curves_BiLSTM_Attention.png'))

## 6. Evaluate on Test Set

In [ ]:
test_preds, test_probas, test_attns = predict_with_attention(
    model, test_texts, vocab, max_len=256, batch_size=64, device=DEVICE,
)

metrics = evaluate_model(
    np.array(test_labels), test_preds, test_probas,
    label_names=label_names,
    model_name='bilstm_attention_product',
)

print(f"Test Results:")
print(f"  Accuracy:    {metrics['accuracy']:.4f}")
print(f"  Macro F1:    {metrics['f1_macro']:.4f}")
print(f"  Weighted F1: {metrics['f1_weighted']:.4f}")
print(f"\n{metrics['classification_report']}")

In [ ]:
plot_confusion_matrix(
    np.array(test_labels), test_preds,
    label_names=label_names,
    model_name='bilstm_attention_product',
)
Image(str(RESULTS_DIR / 'confusion_matrix_bilstm_attention_product.png'))

## 7. Attention Weight Visualization

In [ ]:
def visualize_attention(tokens, weights, pred_label, true_label, confidence, ax=None):
    """Render word-level attention as a heatmap on tokens."""
    # Clip to first 30 tokens for readability
    tokens  = tokens[:30]
    weights = weights[:30]
    weights = weights / (weights.max() + 1e-9)

    if ax is None:
        fig, ax = plt.subplots(figsize=(14, 1.5))

    cmap = plt.cm.YlOrRd
    for i, (tok, w) in enumerate(zip(tokens, weights)):
        ax.text(i, 0.5, tok,
                ha='center', va='center',
                fontsize=9, fontweight='bold',
                color='black' if w < 0.6 else 'white',
                bbox=dict(facecolor=cmap(w), edgecolor='none',
                          boxstyle='round,pad=0.3'))

    ax.set_xlim(-0.5, len(tokens) - 0.5)
    ax.set_ylim(0, 1)
    ax.axis('off')
    title = f'Pred: {pred_label} (conf={confidence:.2f}) | True: {true_label}'
    ax.set_title(title, fontsize=9, fontweight='bold',
                 color='green' if pred_label == true_label else 'red')

print('Attention visualization helper ready.')

In [ ]:
# Visualize attention for 5 test examples
import random
random.seed(42)
sample_indices = random.sample(range(len(test_texts)), 5)

fig, axes = plt.subplots(5, 1, figsize=(16, 10))

for ax, idx in zip(axes, sample_indices):
    tokens, attn, proba = get_attention_for_text(
        model, test_texts[idx], vocab, max_len=256, device=DEVICE,
    )
    pred_class = proba.argmax()
    pred_label = label_names[pred_class]
    true_label = label_names[test_labels[idx]]
    confidence = proba.max()

    visualize_attention(tokens, attn, pred_label, true_label, confidence, ax=ax)

plt.suptitle('BiLSTM Attention Weights (Token Importance for Classification)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'attention_visualization.png', dpi=150, bbox_inches='tight')
plt.show()
print('Darker = higher attention weight (more important token)')

## 8. Summary

The BiLSTM + Attention model:
- **Improves** over the baseline by capturing sequential context and word order.
- **Attention visualization** shows which words drive the classification decision.
- **Limitations:** Fixed vocabulary, no subword tokenization, limited context window.

Results saved to `results/bilstm_attention_product_metrics.json`.
Best model checkpoint saved to `models/bilstm_attention.pt`.